# 04. Portugal vs Spain vs EU average
Are fuel prices in Portugal higher than in Spain and in the EU? And how much of the difference comes from taxes, and how much from the price before taxes?

In [2]:
# Import the libraries
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Load prices with and without taxes from the Oil Bulletin
file = "../data/raw/oil_bulletin_history.xlsx"
with_tax = pd.read_excel(file, sheet_name="Prices with taxes")
without_tax = pd.read_excel(file, sheet_name="Prices wo taxes")
with_tax = with_tax.rename(columns={with_tax.columns[0]: "date"})
without_tax = without_tax.rename(columns={without_tax.columns[0]: "date"})

# Countries and fuels to compare (EU = EU average published by the European Commission)
countries = ["PT", "ES", "EU"]
fuels = {"gasoline": "euro95", "diesel": "diesel"}

# Original column names and new simple names, e.g. "ES_price_with_tax_diesel" becomes "es_diesel_with_tax"
with_columns = {"date": "date"}
without_columns = {"date": "date"}
for country in countries:
    for fuel_name, code in fuels.items():
        with_columns[f"{country}_price_with_tax_{code}"] = f"{country.lower()}_{fuel_name}_with_tax"
        without_columns[f"{country}_price_wo_tax_{code}"] = f"{country.lower()}_{fuel_name}_without_tax"

# Keep only those columns, rename them and merge both sheets by date
with_tax = with_tax[list(with_columns.keys())].rename(columns=with_columns)
without_tax = without_tax[list(without_columns.keys())].rename(columns=without_columns)
compare = pd.merge(with_tax, without_tax, on="date")

# Real dates, numbers in euros per litre, from 2019 to September 2026
compare["date"] = pd.to_datetime(compare["date"], errors="coerce", format="mixed")
for col in compare.columns:
    if col != "date":
        compare[col] = pd.to_numeric(compare[col], errors="coerce") / 1000
compare = compare.dropna()
compare = compare[(compare["date"] >= "2019-01-01") & (compare["date"] <= "2026-09-28")]
compare = compare.sort_values("date")

# Taxes per litre = price with taxes minus price without taxes
for country in ["pt", "es", "eu"]:
    for fuel_name in ["gasoline", "diesel"]:
        compare[f"{country}_{fuel_name}_taxes"] = compare[f"{country}_{fuel_name}_with_tax"] - compare[f"{country}_{fuel_name}_without_tax"]

# Save and check
compare.to_csv("../data/processed/portugal_spain_eu.csv", index=False)
print(len(compare), "weeks")
compare.tail()

395 weeks


,date,pt_gasoline_with_tax,pt_diesel_with_tax,es_gasoline_with_tax,es_diesel_with_tax,eu_gasoline_with_tax,eu_diesel_with_tax,pt_gasoline_without_tax,pt_diesel_without_tax,es_gasoline_without_tax,es_diesel_without_tax,eu_gasoline_without_tax,eu_diesel_without_tax,pt_gasoline_taxes,pt_diesel_taxes,es_gasoline_taxes,es_diesel_taxes,eu_gasoline_taxes,eu_diesel_taxes
30,2026-08-31,2.013,2.029,1.732494,1.865410,1.949992,2.039147,1.031578,1.186283,1.059123,1.262661,1.016038,1.218685,0.981422,0.842717,0.673371,0.602749,0.933954,0.820462
29,2026-09-07,2.093,2.104,1.815973,1.795450,2.043028,2.108413,1.120478,1.272119,1.078114,1.304843,1.070448,1.272249,0.972522,0.831881,0.737859,0.490607,0.972580,0.836164
28,2026-09-14,2.072,2.152,1.866893,1.834025,2.063358,2.158736,1.093845,1.312233,1.120196,1.336723,1.087834,1.315311,0.978155,0.839767,0.746696,0.497302,0.975524,0.843425
27,2026-09-21,2.107,2.210,1.927981,1.918065,2.092159,2.226444,1.130600,1.370627,1.170683,1.406178,1.111631,1.365943,0.976400,0.839373,0.757298,0.511887,0.980529,0.860501
26,2026-09-28,2.099,2.186,1.939726,1.934427,2.103792,2.237184,1.123306,1.342585,1.180389,1.419700,1.121541,1.375062,0.975694,0.843415,0.759337,0.514727,0.982252,0.862122


In [4]:
# Average pump price, price without taxes and taxes per litre, by country (2019 to 2026)
summary = []
for fuel_name in ["gasoline", "diesel"]:
    for country in ["pt", "es", "eu"]:
        summary.append({
            "fuel": fuel_name,
            "country": country.upper(),
            "pump_price": round(compare[f"{country}_{fuel_name}_with_tax"].mean(), 3),
            "without_taxes": round(compare[f"{country}_{fuel_name}_without_tax"].mean(), 3),
            "taxes": round(compare[f"{country}_{fuel_name}_taxes"].mean(), 3),
        })

summary = pd.DataFrame(summary)
summary.to_csv("../data/processed/portugal_spain_eu_summary.csv", index=False)
summary

,fuel,country,pump_price,without_taxes,taxes
0,gasoline,PT,1.668,0.744,0.924
1,gasoline,ES,1.492,0.772,0.720
2,gasoline,EU,1.610,0.727,0.883
3,diesel,PT,1.551,0.792,0.759
4,diesel,ES,1.427,0.812,0.615
5,diesel,EU,1.541,0.791,0.750


In [5]:
# Where does the gap between Portugal and Spain come from? (average, cents per litre)
for fuel_name in ["gasoline", "diesel"]:
    total_gap = (compare[f"pt_{fuel_name}_with_tax"] - compare[f"es_{fuel_name}_with_tax"]).mean() * 100
    tax_gap = (compare[f"pt_{fuel_name}_taxes"] - compare[f"es_{fuel_name}_taxes"]).mean() * 100
    market_gap = (compare[f"pt_{fuel_name}_without_tax"] - compare[f"es_{fuel_name}_without_tax"]).mean() * 100
    print(f"{fuel_name}: Portugal is {total_gap:.1f} cents more expensive than Spain, of which {tax_gap:.1f} from taxes and {market_gap:.1f} from the price before taxes")

gasoline: Portugal is 17.6 cents more expensive than Spain, of which 20.5 from taxes and -2.8 from the price before taxes
diesel: Portugal is 12.3 cents more expensive than Spain, of which 14.4 from taxes and -2.0 from the price before taxes
